In [1]:
import pandas as pd

In [2]:
orderbook = pd.read_csv('orderbook_train.csv')
public = pd.read_csv('public_trades_train.csv')

In [3]:
orderbook.columns

Index(['timestamp', 'ask_1_price', 'ask_1_size', 'bid_1_price', 'bid_1_size',
       'ask_2_price', 'ask_2_size', 'bid_2_price', 'bid_2_size', 'ask_3_price',
       'ask_3_size', 'bid_3_price', 'bid_3_size', 'ask_4_price', 'ask_4_size',
       'bid_4_price', 'bid_4_size', 'ask_5_price', 'ask_5_size', 'bid_5_price',
       'bid_5_size'],
      dtype='object')

In [4]:
public.columns

Index(['timestamp', 'price', 'size', 'side'], dtype='object')

In [ ]:
import pandas as pd
import numpy as np

class AutomatedMarketMaking:
    def __init__(self, tick_size=0.1, lot_size=2):
        self.tick_size  = tick_size
        self.lot_size   = lot_size
        self.reset_simulator()

    def reset_simulator(self):
        self.inventory  = 0
        self.active_bid = None
        self.active_ask = None
        self.valid_from = None

    def update_quote(self, timestamp, bid_price, ask_price):
        # Post or update your quote at timestamp It takes effect at t+1
        self.active_bid = bid_price
        self.active_ask = ask_price
        self.valid_from = timestamp + 1

    def process_trades(self, timestamp, trades_at_t):
        # Process all public trades at timestamp Returns updated inventory
        if self.valid_from is None or timestamp < self.valid_from:
            return self.inventory

        filled = False

        # sellside fill against your bid
        sells = trades_at_t[trades_at_t.side == 'sell']
        if self.active_bid is not None and not sells.empty:
            if self.active_bid >= sells.price.max():
                self.inventory += self.lot_size
                self.active_bid = None
                filled = True

        # buyside fill against your ask
        buys = trades_at_t[trades_at_t.side == 'buy']
        if self.active_ask is not None and not buys.empty:
            if self.active_ask <= buys.price.min():
                self.inventory -= self.lot_size
                self.active_ask = None
                filled = True

        if filled:
            # deactivate until next update
            self.valid_from = float('inf')

        return self.inventory

    def strategy(self, ob_df, tr_df, inventory, t):
        ob_past = ob_df[ob_df.timestamp <= t]
    
        # Safety check
        if ob_past.empty:
            return None, None

        # Get the current orderbook row
        row = ob_past[ob_past.timestamp == t].iloc[-1]

        bid1, ask1 = row['bid_1_price'], row['ask_1_price']
        mid_price = (bid1 + ask1) / 2

        # Estimate volatility from past mid-prices
        mid_prices = (ob_past['bid_1_price'] + ob_past['ask_1_price']) / 2
        sigma = np.std(mid_prices[-20:]) if len(mid_prices) > 10 else 0.1  # avoid zero

        # Parameters
        gamma = 0.1  # risk aversion
        q = self.lot_size

        s = (gamma * sigma * q)*10
        r = mid_price - gamma * sigma**2 * inventory

        raw_bid = r - s/2
        raw_ask = r + s/2

        # Enforce tick size and bid < ask
        bid = max(self.tick_size, round(raw_bid / self.tick_size) * self.tick_size)
        ask = max(bid + self.tick_size, round(raw_ask / self.tick_size) * self.tick_size)

        # Inventory limits
        if inventory >= 20:
            bid = None
        if inventory <= -20:
            ask = None

        return bid, ask

    def run(self, ob_df, tr_df):
        self.reset_simulator()
        quotes = []

        all_ts = sorted(ob_df.timestamp.unique())
        for t in all_ts:
            trades_t = tr_df[tr_df.timestamp == t]
            inv      = self.process_trades(t, trades_t)

            bid, ask = self.strategy(ob_df, tr_df, inv, t)

            self.update_quote(t, bid, ask)

            quotes.append({
                'timestamp': t,
                'bid_price': bid,
                'ask_price': ask
            })

        return pd.DataFrame(quotes)

if __name__ == "__main__":
    ob_obj = pd.read_csv('orderbook_train.csv')
    tr_obj = pd.read_csv('public_trades_train.csv')
    
    #pick top 3k timestamps
    ob_obj = ob_obj.head(3000); 
    tr_obj = tr_obj.head(3000);

    amm = AutomatedMarketMaking(tick_size=0.1, lot_size=2)

    df_submission = amm.run(ob_obj, tr_obj)
    df_submission.to_csv('submission.csv', index=False)

: 

In [75]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import brentq
from scipy.interpolate import RegularGridInterpolator
import io
import time  # Import the time module

# --- 1. IMPLIED VOLATILITY CALCULATION ---

# Constants
spot_price = 100
risk_free_rate = 0.05

# Option data
option_data = [
    # DTC
    (50, 1, 52.44), (50, 2, 54.77), (50, 5, 61.23),
    (75, 1, 28.97), (75, 2, 33.04), (75, 5, 43.47),
    (100, 1, 10.45), (100, 2, 16.13), (100, 5, 29.14),
    (125, 1, 2.32), (125, 2, 6.54), (125, 5, 18.82),
    (150, 1, 0.36), (150, 2, 2.34), (150, 5, 11.89),
    # DFC
    (50, 1, 52.45), (50, 2, 54.90), (50, 5, 61.87),
    (75, 1, 29.11), (75, 2, 33.34), (75, 5, 43.99),
    (100, 1, 10.45), (100, 2, 16.13), (100, 5, 29.14),
    (125, 1, 2.80), (125, 2, 7.39), (125, 5, 20.15),
    (150, 1, 1.26), (150, 2, 4.94), (150, 5, 17.46),
    # DEC
    (50, 1, 52.44), (50, 2, 54.80), (50, 5, 61.42),
    (75, 1, 29.08), (75, 2, 33.28), (75, 5, 43.88),
    (100, 1, 10.45), (100, 2, 16.13), (100, 5, 29.14),
    (125, 1, 1.96), (125, 2, 5.87), (125, 5, 17.74),
    (150, 1, 0.16), (150, 2, 1.49), (150, 5, 9.70)
]

stocks = ['DTC'] * 15 + ['DFC'] * 15 + ['DEC'] * 15
data = []

def black_scholes_call_price(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + sigma ** 2 / 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def implied_volatility(S, K, T, r, market_price):
    try:
        return brentq(lambda sigma: black_scholes_call_price(S, K, T, r, sigma) - market_price,
                      1e-6, 5.0, maxiter=500)
    except ValueError:
        return np.nan

# Calculate implied volatilities
for (K, T, price), stock in zip(option_data, stocks):
    iv = implied_volatility(spot_price, K, T, risk_free_rate, price)
    data.append((stock, K, T, price, iv))

# Create DataFrame and pivot to matrix form
df = pd.DataFrame(data, columns=['Stock', 'Strike', 'Maturity', 'Price', 'ImpliedVol'])
vol_matrix = df.pivot_table(index='Strike', columns=['Stock', 'Maturity'], values='ImpliedVol')

# Extract implied volatility surfaces for each stock
strikes = sorted(df['Strike'].unique())
maturities = sorted(df['Maturity'].unique())

local_vol_DTC = vol_matrix['DTC'].values
local_vol_DFC = vol_matrix['DFC'].values
local_vol_DEC = vol_matrix['DEC'].values

vol_surfaces = [local_vol_DTC, local_vol_DFC, local_vol_DEC]

def get_interpolators(vol_surface):
    return RegularGridInterpolator((strikes, maturities), vol_surface)

# Pre-compute interpolators
interpolators = [get_interpolators(vol) for vol in vol_surfaces]

# --- 2. PARAMETERS FOR MONTE CARLO ---

spot_prices = np.array([100, 100, 100])  # DTC, DFC, DEC
corr_matrix = np.array([
    [1.0, 0.8, 0.75],
    [0.8, 1.0, 0.85],
    [0.75, 0.85, 1.0]
])

# Pre-compute Cholesky decomposition
chol = np.linalg.cholesky(corr_matrix)

# --- 3. MONTE CARLO SIMULATION ---

def simulate_basket_price(knockout, maturity_yrs, strike, option_type,
                         num_paths=10000, steps_per_year=252):
    dt = 1 / steps_per_year
    steps = int(maturity_yrs * steps_per_year)
    dim = len(spot_prices)

    prices = np.full((num_paths, dim), spot_prices, dtype=np.float64)

    basket_prices = np.zeros((num_paths, steps + 1))
    basket_prices[:, 0] = np.mean(prices, axis=1)

    for t in range(1, steps + 1):
        Z = np.random.normal(size=(num_paths, dim))
        correlated_Z = Z @ chol.T

        for i in range(dim):
            S = prices[:, i]
            t_years = t * dt
            # Interpolate implied volatility
            vol = interpolators[i](
                (np.clip(S, min(strikes), max(strikes)), np.full(S.shape, maturity_yrs))
            )
            dS = risk_free_rate * S * dt + vol * S * np.sqrt(dt) * correlated_Z[:, i]
            prices[:, i] += dS

        basket_prices[:, t] = np.mean(prices, axis=1)

    knocked_out = np.any(basket_prices >= knockout, axis=1)
    final_prices = basket_prices[:, -1]

    if option_type.lower() == 'call':
        payoffs = np.maximum(final_prices - strike, 0)
    else:
        payoffs = np.maximum(strike - final_prices, 0)

    payoffs[knocked_out] = 0
    discounted = np.exp(-risk_free_rate * maturity_yrs) * payoffs
    return np.mean(discounted)

# --- 4. EMBEDDED INPUT CSV ---

input_csv = """
Id,Asset,KnockOut,Maturity,Strike,Type
1,Basket,150,2y,50,Call
2,Basket,175,2y,50,Call
3,Basket,200,2y,50,Call
4,Basket,150,5y,50,Call
5,Basket,175,5y,50,Call
6,Basket,200,5y,50,Call
7,Basket,150,2y,100,Call
8,Basket,175,2y,100,Call
9,Basket,200,2y,100,Call
10,Basket,150,5y,100,Call
11,Basket,175,5y,100,Call
12,Basket,200,5y,100,Call
13,Basket,150,2y,125,Call
14,Basket,175,2y,125,Call
15,Basket,200,2y,125,Call
16,Basket,150,5y,125,Call
17,Basket,175,5y,125,Call
18,Basket,200,5y,125,Call
19,Basket,150,2y,75,Put
20,Basket,175,2y,75,Put
21,Basket,200,2y,75,Put
22,Basket,150,5y,75,Put
23,Basket,175,5y,75,Put
24,Basket,200,5y,75,Put
25,Basket,150,2y,100,Put
26,Basket,175,2y,100,Put
27,Basket,200,2y,100,Put
28,Basket,150,5y,100,Put
29,Basket,175,5y,100,Put
30,Basket,200,5y,100,Put
31,Basket,150,2y,125,Put
32,Basket,175,2y,125,Put
33,Basket,200,2y,125,Put
34,Basket,150,5y,125,Put
35,Basket,175,5y,125,Put
36,Basket,200,5y,125,Put
"""

input_df = pd.read_csv(io.StringIO(input_csv.strip()))

# --- 5. PROCESSING ---

output = []
start_time = time.time() # start the timer
for _, row in input_df.iterrows():
    id_, ko, maturity_str, strike, opt_type = row['Id'], row['KnockOut'], row['Maturity'], row['Strike'], row['Type']
    maturity = int(maturity_str.strip('y'))
    price = simulate_basket_price(knockout=ko, maturity_yrs=maturity, strike=strike, option_type=opt_type, num_paths=3000, steps_per_year=52) # Increased num_paths
    output.append((id_, round(price, 2)))

output_df = pd.DataFrame(output, columns=["Id", "Price"])
end_time = time.time()  # End the timer
print(output_df.to_csv(index=False))

Id,Price
1,38.59
2,47.97
3,52.57
4,17.37
5,30.0
6,38.93
7,6.71
8,11.65
9,13.7
10,3.05
11,7.8
12,12.46
13,0.95
14,3.02
15,4.29
16,0.43
17,2.37
18,4.73
19,0.69
20,0.81
21,0.78
22,1.52
23,1.79
24,1.68
25,6.18
26,5.95
27,5.99
28,5.92
29,6.33
30,6.42
31,18.41
32,19.0
33,19.07
34,14.62
35,15.26
36,15.28



In [74]:
import numpy as np
import pandas as pd
import math
import io
from scipy.stats import norm
from scipy.optimize import brentq


_strikes = np.array([50,75,100,125,150], dtype=float)
_maturities = np.array([1.0,2.0,5.0], dtype=float)
_calib_market = {
    'DTC': np.array([[52.44,54.77,61.23], [28.97,33.04,43.47], [10.45,16.13,29.14], [2.32,6.54,18.82], [0.36,2.34,11.89]]),
    'DFC': np.array([[52.45,54.90,61.87], [29.11,33.34,43.99], [10.45,16.13,29.14], [2.80,7.39,20.15], [1.26,4.94,17.46]]),
    'DEC': np.array([[52.44,54.80,61.42], [29.08,33.28,43.88], [10.45,16.13,29.14], [1.96,5.87,17.74], [0.16,1.49,9.70]])
}
_r = 0.05

def bs_price(S, K, T, r, sigma, option_type='call'):
    if T<=0 or sigma<=0:
        return max((S-K) if option_type=='call' else (K-S),0)
    d1 = (math.log(S/K)+(r+0.5*sigma**2)*T)/(sigma*math.sqrt(T))
    d2 = d1 - sigma*math.sqrt(T)
    if option_type=='call':
        return S*norm.cdf(d1)-K*math.exp(-r*T)*norm.cdf(d2)
    return K*math.exp(-r*T)*norm.cdf(-d2)-S*norm.cdf(-d1)

def implied_vol(mkt_price, S, K, T, r, option_type='call'):
    f = lambda vol: bs_price(S,K,T,r,vol,option_type) - mkt_price
    try:
        return brentq(f,1e-6,5.0)
    except:
        return np.nan

_vol_surfaces = {}
for stock, prices in _calib_market.items():
    vols = np.zeros_like(prices)
    for i,K in enumerate(_strikes):
        for j,T in enumerate(_maturities):
            vols[i,j] = implied_vol(prices[i,j], 100.0, K, T, _r, 'call')
    _vol_surfaces[stock] = vols

def local_vol(stock, S, t):
    i = np.searchsorted(_strikes, S)
    i = min(max(i,1), len(_strikes)-1)
    j = np.searchsorted(_maturities, t)
    j = min(max(j,1), len(_maturities)-1)
    return _vol_surfaces[stock][i,j]

class BarrierOption:
    def __init__(self, strike, barrier):
        self.strike = strike
        self.knock_out_price = barrier
        self.knock_in_price = barrier
        self.knocked_out = False
        self.knock_in = False

class BasketBarrierOption(BarrierOption):
    def __init__(self, strike, barrier, maturity, opt_type):
        super().__init__(strike, barrier)
        self.maturity = float(maturity[:-1])
        self.opt_type = opt_type.lower()

class BasketCallBarrierOptionUpandOutSimulation:
    def __init__(self, option, n_paths=10, steps_per_year=50):
        T = option.maturity
        K = option.strike
        barrier = option.knock_out_price
        dt = 1.0/steps_per_year
        n_steps = int(steps_per_year * T)
        corr = np.array([[1.0,0.75,0.5],[0.75,1.0,0.25],[0.5,0.25,1.0]])
        L = np.linalg.cholesky(corr)
        payoffs = np.zeros(n_paths)
        np.random.seed(0)
        for p in range(n_paths):
            knocked = False
            S = np.array([100.0,100.0,100.0])
            t = 0.0
            for _ in range(n_steps):
                z = np.random.normal(size=3)
                dW = L.dot(z)*math.sqrt(dt)
                for idx,stock in enumerate(['DTC','DFC','DEC']):
                    vol = local_vol(stock, S[idx], t)
                    S[idx] += _r*S[idx]*dt + vol*S[idx]*dW[idx]
                basket = S.mean()
                if basket>=barrier:
                    knocked=True
                    break
                t += dt
            if not knocked:
                basket = S.mean()
                pay = max((basket-K) if option.opt_type=='call' else (K-basket), 0)
                payoffs[p] = pay
        self.price = np.exp(-_r*T)*np.mean(payoffs)

strikes = [50, 75, 100, 125]
maturities = ["1y", "2y", "5y"]
knock_out_barriers = [125, 150, 175]
option_types = ["Call", "Put"]

combinations = []
option_id = 1
for strike in strikes:
    for maturity in maturities:
        for barrier in knock_out_barriers:
            for opt_type in option_types:
                combinations.append({
                    'Id': option_id,
                    'Asset': 'Basket',
                    'KnockOut': barrier,
                    'Maturity': maturity,
                    'Strike': strike,
                    'Type': opt_type
                })
                option_id += 1

df = pd.DataFrame(combinations)

results = []
for _, row in df.iterrows():
    opt = BasketBarrierOption(float(row['Strike']), float(row['KnockOut']), row['Maturity'], row['Type'])
    sim = BasketCallBarrierOptionUpandOutSimulation(opt)
    results.append((row['Id'], sim.price))

print('Id,Price')
for Id, p in results:
    if p > 0:
        print(f"{Id},{p}")
    else:
        print(f"{Id}, {p+0.01}")

Id,Price
1,42.42303314808305
2, 0.01
3,49.97611402364075
4, 0.01
5,49.97611402364075
6, 0.01
7,29.57026879376423
8, 0.01
9,41.137376074987024
10, 0.01
11,49.47012832313425
12, 0.01
13,9.246618041880007
14, 0.01
15,16.15932262791488
16, 0.01
17,37.1510749613567
18, 0.01
19,21.02037109681699
20, 0.01
21,26.195378411122896
22, 0.01
23,26.195378411122896
24, 0.01
25,13.735613978134941
26, 0.01
27,20.973945356017996
28,0.1954111868400612
29,26.84919287223526
30, 0.01
31,2.2226502228663505
32,0.7640400117003928
33,6.496367756427002
34,2.019056874583197
35,20.201696292841067
36,0.5736389505909774
37,4.082987460694773
38,4.465278415143854
39,7.4591315430996135
40,5.0444887444945685
41,7.4591315430996135
42,5.0444887444945685
43,1.8396433187119703
44,3.938684156206321
45,6.511716132945612
46,6.092023869576767
47,11.927197028117108
48,7.698939606780834
49,0.2756482651878381
50,6.6050458847359295
51,1.8030211298374075
52,9.007721994064674
53,7.704951689102003
54,5.599911965958522
55, 0.01
56,21.7